In [1]:
import csv
import json
import subprocess
import numpy as np
from tqdm import tqdm
import networkx as nx
from pathlib import Path
from scipy.optimize import brentq

cwd = Path.cwd()
INDEX_JSONL = str(cwd / "index-nx.jsonl")
with open(INDEX_JSONL, "r") as fin:
    lines = sum(1 for _ in fin)

In [2]:
def bounds1(A):
    A = np.asarray(A, dtype=float)
    spec_A = np.linalg.eigvalsh(A)
    n = A.shape[0]
    k = spec_A[-1]
    sig = k * np.sqrt(np.sum((k - spec_A[:-1]) ** (-2)))
    ups = np.sqrt(k * (n - k))
    rprmax, rppmax = 0.0, 0.0
    if sig * ups / k <= 1:
        rprmax = 1 / (sig * ups)
        rppmax = np.inf
    else:
        rprmax = brentq(
            lambda r: (r - 1 / (sig * ups) + (1 / k - r) * np.tanh(r / np.sqrt(n)) ** 2),
            0.0,
            1 / (sig * ups),
            xtol=np.finfo(float).tiny,
            rtol=8 * np.finfo(float).eps,
        )
        rppmax = np.sqrt(n / (n - 1)) * np.arctanh(1.0 / np.sqrt(sig * ups / k))
    return rprmax, rppmax


with open(INDEX_JSONL, "r") as fin, open(str(cwd / "bounds1.csv"), "w", newline="") as fout:
    writer = csv.writer(fout)
    writer.writerow(["n", "k", "avg_rprmax", "avg_rppmax"])

    for line in tqdm(fin, total=lines, ncols=128):
        if not line.strip():
            continue

        record = json.loads(line)

        n = record["n"]
        k = record["k"]
        graphs = record["graphs"]

        total_rpr, total_rpp = 0.0, 0.0

        for g6 in graphs:
            G = nx.from_graph6_bytes(g6.encode("ascii"))
            A = nx.to_numpy_array(G, dtype=float)
            rpr, rpp = bounds1(A)
            total_rpr += rpr
            total_rpp += rpp

        avg_rpr = total_rpr / len(graphs) if graphs else float("nan")
        avg_rpp = total_rpp / len(graphs) if graphs else float("nan")
        writer.writerow([n, k, avg_rpr, avg_rpp])

100%|██████████████████████████████████████████████████████████████████████████████████████| 1488/1488 [00:12<00:00, 123.73it/s]


In [3]:
def bounds2approx(G):
    d = 1.0
    m1 = 1.0
    m2 = 4/(3*np.sqrt(3))
    n = len(G)
    A = nx.to_numpy_array(G, dtype=int)
    eigvals = np.linalg.eigvalsh(A)
    k = eigvals[-1]
    Mpp = k * np.sqrt(np.sum((k - eigvals[:-1])**(-2)))

    def opt(R):
        kuu = np.sqrt(k * (n - k)) * np.sqrt(((d / k + R) ** 2 / n) * m2**2 + 2 * m1**2)
        kuv = np.sqrt(k * (n - k)) * np.sqrt(((d / k + R) ** 2 / n) * m2**2 + m1**2)
        kvv = m2 * (d / k + R) * np.sqrt(k * (n - k) * (1 - 1 / n))
        return R - 1/(Mpp * (kuv + np.sqrt(kuu*kvv)))
    
    R = 0.1
    while opt(R) < 0:
        R *= 2
        if R > 1e12:
            raise ValueError("Could not bracket the root.")
    
    rprmax = brentq(opt, 0.0, R)    
    rppmax = 1/(Mpp * m2 * (d / k) * np.sqrt(k * (n - k) * (1 - 2 / n)))

    return rprmax, rppmax

with open(INDEX_JSONL, "r") as fin, open("bounds2approx.csv", "w", newline="") as fout:
    writer = csv.writer(fout)
    writer.writerow(["n", "k", "avg_rprmax", "avg_rppmax"])

    for line in tqdm(fin, total=lines, ncols=128):
        if not line.strip():
            continue

        record = json.loads(line)

        n = record["n"]
        k = record["k"]
        graphs = record["graphs"]

        total_rpr, total_rpp = 0.0, 0.0

        for g6 in graphs:
            G = nx.from_graph6_bytes(g6.encode("ascii"))
            rpr, rpp = bounds2approx(G)
            total_rpr += rpr
            total_rpp += rpp

        avg_rpr = total_rpr / len(graphs) if graphs else float("nan")
        avg_rpp = total_rpp / len(graphs) if graphs else float("nan")
        writer.writerow([n, k, avg_rpr, avg_rpp])

100%|██████████████████████████████████████████████████████████████████████████████████████| 1488/1488 [00:11<00:00, 129.21it/s]


In [4]:
def bounds2true(G):
    n = len(G)
    A = nx.to_numpy_array(G, dtype=int)
    g6 = nx.to_graph6_bytes(G, header=False).decode("ascii").strip()
    cmd = [str(Path.cwd() / "rmax"), g6]
    result = subprocess.run(cmd, capture_output=True, text=True)
    rmax = result.stdout.strip().splitlines()[-1]
    rprmax, rppmax = map(float, rmax.split())
    return rprmax, rppmax

with open(INDEX_JSONL, "r") as fin, open("bounds2true.csv", "w", newline="") as fout:
    writer = csv.writer(fout)
    writer.writerow(["n", "k", "avg_rprmax", "avg_rppmax"])

    for line in tqdm(fin, total=lines, ncols=128):
        if not line.strip():
            continue

        record = json.loads(line)

        n = record["n"]
        k = record["k"]
        graphs = record["graphs"]

        total_rpr, total_rpp = 0.0, 0.0

        for g6 in graphs:
            G = nx.from_graph6_bytes(g6.encode("ascii"))
            rpr, rpp = bounds2true(G)
            total_rpr += rpr
            total_rpp += rpp

        avg_rpr = total_rpr / len(graphs) if graphs else float("nan")
        avg_rpp = total_rpp / len(graphs) if graphs else float("nan")
        writer.writerow([n, k, avg_rpr, avg_rpp])

100%|███████████████████████████████████████████████████████████████████████████████████████| 1488/1488 [12:33<00:00,  1.97it/s]
